# Revision de los datos de restauracion

Fase de mirar y decidir. **Este cuaderno no escribe ningun CSV**: cuando todo este cerrado, se
pasa a un script y se genera el `gold`.

## La fuente: solo el censo municipal

De las tres que habia sobre la mesa, se descartan dos:

| Fuente | Decision | Motivo |
|---|---|---|
| **Censo comercial del Ajuntament** | **se usa** | foto con fecha, 31/12/2024 |
| OpenStreetMap | descartada en la ciudad | cuenta 7.430 locales frente a 10.100 del censo, un 26% menos, y el fichero no trae fecha de cada elemento |
| Terrazas | descartada | solo mide las sillas de la calle, no el aforo total: un bar con 8 sillas fuera puede tener 60 plazas dentro |

El censo es de **2024 y no hay nada mas nuevo**: la serie publicada es 2014, 2016, 2019, 2022 y
2024, confirmado en la web del Ajuntament, en Open Data BCN y en el portal estadistico. Para 2026
esta anunciado el IDUL, un identificador unico de local; si sale, cruzar el censo con cualquier otro
registro municipal dejara de ser un problema.


In [1]:
# 1. Carga
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parents[1] if Path.cwd().name == "notebooks" else Path.cwd()
RUTA = (RAIZ / "data" / "raw" / "restauracion_hoteles_provincia" /
        "bcn_cens_comercial_restauracion_2024.csv")

censo = pd.read_csv(RUTA, low_memory=False)
print(f"{len(censo):,} locales x {censo.shape[1]} columnas")
print(f"grupo de actividad: {censo['Nom_Grup_Activitat'].unique()}")

10,864 locales x 49 columnas
grupo de actividad: <StringArray>
['Restaurants, bars i hotels (Inclòs hostals, pensions i fondes)']
Length: 1, dtype: str


## 1. Que categorias trae, y cuales son restauracion

El fichero viene filtrado por grupo de actividad, y ese grupo mete en el mismo saco la restauracion
y el alojamiento. Dentro hay mas variedad de la que parece.

In [2]:
# 2. Todas las categorias, sin filtrar nada todavia
reparto = censo["Nom_Activitat"].value_counts().rename("locales").to_frame()
reparto["%"] = (reparto["locales"] / len(censo) * 100).round(1)
display(reparto)

,locales,%
Nom_Activitat,,
Restaurants,4430,40.8
Bars / CIBERCAFÈ,4273,39.3
Serveis de menjar take away MENJAR RÀPID,778,7.2
serveis d'allotjament,764,7.0
Bars especials amb actuació / Bars musicals / Discoteques /PUB,387,3.6
Xocolateries / Geladeries / Degustació,148,1.4
serveis de menjar i begudes,74,0.7
altres,6,0.1
Altres ( per exemple VENDING),4,0.0


In [3]:
# 3. Que hay dentro de las categorias dudosas
#
# Los recuentos no bastan para decidir: hay que ver que locales son. Tres de estas categorias no
# son restauracion aunque esten en el mismo grupo de actividad.
for actividad in ["serveis de menjar i begudes", "altres", "Altres ( per exemple VENDING)"]:
    g = censo[censo["Nom_Activitat"] == actividad]
    print(f"-- {actividad}  ({len(g)} locales)")
    for nombre in g["Nom_Local"].head(6):
        print(f"     {nombre}")
    print()

-- serveis de menjar i begudes  (74 locales)
     BANCO DE BOQUERONES
     SN
     SN
     SN
     SN
     SN

-- altres  (6 locales)
     CHIQUITA ROOM
     SN
     LOTERIAS Y APUESTAS ADMIN NUM 287
     LOTERIAS Y APUESTAS DEL ESTADO 236
     NONETES
     LOTERIAS Y APUESTAS DEL ESTADO

-- Altres ( per exemple VENDING)  (4 locales)
     ABIERTO 25 HORAS
     SN
     BCN 24 HORES 
     ABIERTO 25 HORAS



## 2. La seleccion

`INCLUIDOS` es la unica linea que hay que tocar para cambiar el alcance. Se declara aqui arriba y
no repartida por el cuaderno, para que no haya dos sitios donde mirar.

In [4]:
# 4. Que entra y que no
#
# Entran bares, restaurantes y comida rapida: con esos tres queda cubierta la restauracion de
# comidas y cenas, que es el objeto del analisis.
#
# Quedan FUERA, cada uno por su motivo:
#   allotjament          -> son hoteles, cuentan en el otro lado del analisis
#   altres               -> administraciones de loteria, no es restauracion
#   Altres (VENDING)     -> tiendas de 24 horas y maquinas
#   menjar i begudes     -> cajon de sastre: 73 de sus 74 locales no tienen ni nombre
#
# Y quedan fuera POR DECISION, no por ser otra cosa. Se listan aparte porque son restauracion de
# pleno derecho y volver a meterlas es cambiar una linea:
#   bars musicals / discoteques / pub   387 locales   -> se bebe, no se cena
#   xocolateries / geladeries           148 locales   -> no cubren una comida

# Las categorias NO se escriben literales. El censo las escribe con espacios dobles y acentos
# --`Bars   / CIBERCAFÉ`-- y una comparacion literal falla en silencio: la primera version de esta
# celda dejaba fuera los 4.273 bares y el cuaderno terminaba sin quejarse. Se buscan por palabra
# clave y se comprueba que cada una encuentre exactamente una categoria.
import unicodedata


def buscar(clave: str) -> str:
    """Devuelve la categoria del censo que contiene `clave`. Falla si no hay exactamente una."""
    def limpiar(s):
        s = unicodedata.normalize("NFKD", str(s).lower())
        return "".join(c for c in s if not unicodedata.combining(c))

    encontradas = [a for a in censo["Nom_Activitat"].unique() if clave in limpiar(a)]
    if len(encontradas) != 1:
        raise ValueError(f"'{clave}' encuentra {len(encontradas)} categorias: {encontradas}")
    return encontradas[0]


INCLUIDOS = {buscar("restaurants"): "restaurante",
             buscar("cibercafe"): "bar",
             buscar("take away"): "comida_rapida"}

OPCIONALES = {buscar("discoteques"): "ocio_nocturno",
              buscar("geladeries"): "degustacion"}

d = censo[censo["Nom_Activitat"].isin(INCLUIDOS)].copy()
d["tipo_local"] = d["Nom_Activitat"].map(INCLUIDOS)

print("=== SELECCION ===")
for etiqueta, n in d["tipo_local"].value_counts().items():
    print(f"  {etiqueta:<14} {n:>5,}")
print(f"  {'TOTAL':<14} {len(d):>5,}")
print()
print("Fuera por decision, se reincorporan anadiendolas a INCLUIDOS:")
for actividad, etiqueta in OPCIONALES.items():
    print(f"  {etiqueta:<14} {int((censo['Nom_Activitat'] == actividad).sum()):>5,}")
print()
print(f"Fuera por no ser restauracion o ser alojamiento: "
      f"{len(censo) - len(d) - sum((censo['Nom_Activitat'] == a).sum() for a in OPCIONALES):,}")

=== SELECCION ===
  restaurante    4,430
  bar            4,273
  comida_rapida    778
  TOTAL          9,481

Fuera por decision, se reincorporan anadiendolas a INCLUIDOS:
  ocio_nocturno    387
  degustacion      148

Fuera por no ser restauracion o ser alojamiento: 848


## 3. Como esta el dato seleccionado

Antes de construir nada: que falta, que se repite y que no cuadra.

In [5]:
# 5. Nulos, y el marcador que no es nulo
#
# OJO: el censo NO deja celdas vacias en el nombre. Usa el literal `SN` --sense nom--, asi que un
# `isna()` dice que estan todos informados y es mentira.
campos = ["Nom_Local", "Nom_Activitat", "Nom_Barri", "Nom_Districte", "Nom_Via",
          "Num_Policia_Inicial", "Latitud", "Longitud", "Data_Revisio"]
tabla = pd.DataFrame({
    "nulos": [int(d[c].isna().sum()) for c in campos],
    "informados_%": [round(d[c].notna().mean() * 100, 1) for c in campos],
}, index=campos)
display(tabla)

sin_nombre = d["Nom_Local"].astype(str).str.strip().str.upper().eq("SN")
print(f"nombre = 'SN' (sin nombre): {int(sin_nombre.sum())} ({sin_nombre.mean():.1%})")
print()
print("Por tipo:")
print(d.assign(sn=sin_nombre).groupby("tipo_local")["sn"].agg(["size", "sum"]).to_string())

,nulos,informados_%
Nom_Local,0,100.0
Nom_Activitat,0,100.0
Nom_Barri,0,100.0
Nom_Districte,0,100.0
Nom_Via,0,100.0
Num_Policia_Inicial,0,100.0
Latitud,0,100.0
Longitud,0,100.0
Data_Revisio,0,100.0


nombre = 'SN' (sin nombre): 56 (0.6%)

Por tipo:
               size  sum
tipo_local              
bar            4273   36
comida_rapida   778    1
restaurante    4430   19


In [6]:
# 6. Duplicados y cobertura territorial
print("=== IDENTIDAD ===")
print(f"filas                : {len(d):,}")
print(f"ID_Global distintos  : {d['ID_Global'].nunique():,}")
print(f"ID_Bcn_2016 repetidos: {int(d['ID_Bcn_2016'].duplicated().sum()):,}")
print()
# Un mismo portal puede tener varios locales: no es un duplicado, son negocios distintos.
misma_direccion = d.duplicated(subset=["Nom_Via", "Num_Policia_Inicial"], keep=False)
print(f"locales que comparten portal con otro: {int(misma_direccion.sum()):,}")
print(f"mismo portal Y mismo nombre           : "
      f"{int(d.duplicated(subset=['Nom_Via', 'Num_Policia_Inicial', 'Nom_Local']).sum()):,}"
      "   <- estos si son sospechosos")
print()
print("=== TERRITORIO ===")
print(f"barrios   : {d['Nom_Barri'].nunique()} de 73")
print(f"distritos : {d['Nom_Districte'].nunique()} de 10")
fuera = ((d["Latitud"] < 41.30) | (d["Latitud"] > 41.48) |
         (d["Longitud"] < 2.05) | (d["Longitud"] > 2.24))
print(f"coordenadas fuera de Barcelona: {int(fuera.sum())}")

=== IDENTIDAD ===
filas                : 9,481
ID_Global distintos  : 9,481
ID_Bcn_2016 repetidos: 684

locales que comparten portal con otro: 1,615
mismo portal Y mismo nombre           : 17   <- estos si son sospechosos

=== TERRITORIO ===
barrios   : 73 de 73
distritos : 10 de 10
coordenadas fuera de Barcelona: 0


In [7]:
# 7. Reparto por distrito y por barrio
por_distrito = (d.groupby("Nom_Districte")
                .agg(locales=("ID_Global", "size"),
                     bares=("tipo_local", lambda s: int((s == "bar").sum())),
                     restaurantes=("tipo_local", lambda s: int((s == "restaurante").sum())))
                .sort_values("locales", ascending=False))
por_distrito["bares_por_restaurante"] = (por_distrito.bares / por_distrito.restaurantes).round(2)
print("=== POR DISTRITO ===")
display(por_distrito)

print("=== LOS DIEZ BARRIOS CON MAS LOCALES ===")
display(d.groupby(["Nom_Districte", "Nom_Barri"])
        .agg(locales=("ID_Global", "size"),
             bares=("tipo_local", lambda s: int((s == "bar").sum())),
             restaurantes=("tipo_local", lambda s: int((s == "restaurante").sum())))
        .sort_values("locales", ascending=False).head(10))

=== POR DISTRITO ===


,locales,bares,restaurantes,bares_por_restaurante
Nom_Districte,,,,
Eixample,2594,990,1432,0.69
Sant Martí,1284,690,498,1.39
Ciutat Vella,1229,402,660,0.61
Sants-Montjuïc,904,421,414,1.02
Sarrià-Sant Gervasi,710,258,365,0.71
Gràcia,634,276,289,0.96
Sant Andreu,616,354,222,1.59
Nou Barris,599,393,173,2.27
Horta-Guinardó,461,304,136,2.24


=== LOS DIEZ BARRIOS CON MAS LOCALES ===


locales  bares  \
Nom_Districte       Nom_Barri                                               
Eixample            la Dreta de l'Eixample                     702    244   
                    l'Antiga Esquerra de l'Eixample            635    192   
Gràcia              la Vila de Gràcia                          437    175   
Ciutat Vella        el Raval                                   410    158   
Eixample            la Nova Esquerra de l'Eixample             373    158   
                    la Sagrada Família                         353    160   
                    Sant Antoni                                349    150   
Sarrià-Sant Gervasi Sant Gervasi - Galvany                     347    109   
Ciutat Vella        el Barri Gòtic                             327     88   
                    Sant Pere, Santa Caterina i la Ribera      315    116   

                                                           restaurantes  
Nom_Districte       Nom_Barri                                            
Eixample            la Dreta de l'Eixample                          413  
                    l'Antiga Esquerra de l'Eixample                 413  
Gràcia              la Vila de Gràcia                               208  
Ciutat Vella        el Raval                                        170  
Eixample            la Nova Esquerra de l'Eixample                  190  
                    la Sagrada Família                              146  
                    Sant Antoni                                     182  
Sarrià-Sant Gervasi Sant Gervasi - Galvany                          189  
Ciutat Vella        el Barri Gòtic                                  196  
                    Sant Pere, Santa Caterina i la Ribera           167

## 4. Que columnas nos dan

49 columnas, pero no todas dicen algo. Se separan por lo que aportan, no por como se llaman.

In [8]:
# 8. Las 49 columnas, clasificadas por lo que aportan
perfil = pd.DataFrame({
    "nulos": [int(d[c].isna().sum()) for c in d.columns],
    "distintos": [int(d[c].nunique(dropna=True)) for c in d.columns],
    "ejemplo": [str(d[c].dropna().iloc[0])[:30] if d[c].notna().any() else "" for c in d.columns],
}, index=d.columns)

# Una columna con un solo valor distinto no distingue nada: ocupa sitio y no informa.
constantes = perfil[perfil["distintos"] <= 1].index.tolist()
print(f"=== CONSTANTES, no aportan nada ({len(constantes)}) ===")
for c in constantes:
    print(f"  {c:<28} siempre '{d[c].iloc[0]}'")
print()
print("  `Nom_Principal_Activitat` vale 'Actiu' en las 9.481: el fichero ya viene filtrado a")
print("  locales en activo, asi que no hay cerrados que descartar.")
print()

# Las que casi siempre estan vacias solo sirven para el puñado que las tiene.
casi_vacias = perfil[(perfil["nulos"] / len(d) > 0.5) & (perfil["distintos"] > 1)]
print(f"=== VACIAS EN MAS DE LA MITAD ({len(casi_vacias)}) ===")
display(casi_vacias)

utiles = perfil.drop(index=constantes + casi_vacias.index.tolist())
print(f"=== CON CONTENIDO EN CASI TODAS LAS FILAS ({len(utiles)}) ===")
display(utiles)


=== CONSTANTES, no aportan nada (7) ===
  Codi_Principal_Activitat     siempre '1'
  Nom_Principal_Activitat      siempre 'Actiu'
  Codi_Sector_Activitat        siempre '2'
  Nom_Sector_Activitat         siempre 'Serveis'
  Codi_Grup_Activitat          siempre '14'
  Nom_Grup_Activitat           siempre 'Restaurants, bars i hotels (Inclòs hostals, pensions i fondes)'
  Planta                       siempre 'LOC'

  `Nom_Principal_Activitat` vale 'Actiu' en las 9.481: el fichero ya viene filtrado a
  locales en activo, asi que no hay cerrados que descartar.

=== VACIAS EN MAS DE LA MITAD (6) ===


,nulos,distintos,ejemplo
Nom_Mercat,9470,10,BOQUERIA
Nom_Galeria,9457,5,CENTRE COMERCIAL DAVID
Nom_CComercial,9288,10,El Triangle
Nom_Eix,6386,24,Rambla Catalunya
Lletra_Inicial,9244,9,X
Lletra_Final,9215,9,B


=== CON CONTENIDO EN CASI TODAS LAS FILAS (37) ===


,nulos,distintos,ejemplo
ID_Global,0,9481,3e40cf2c-3b94-4977-af13-dae686
ID_Bcn_2016,685,8796,44680.0
Codi_Activitat_2022,0,3,1400001
Nom_Activitat,0,3,Bars / CIBERCAFÈ
Codi_Activitat_2016,0,3,1400001
Nom_Local,0,8593,THREE MARKS COFFEE
SN_Oci_Nocturn,0,2,No
SN_Coworking,0,2,No
SN_Servei_Degustacio,0,2,No
SN_Obert24h,0,2,No


In [9]:
# 9. Que hacer con cada bloque
#
# IDENTIDAD      ID_Global (unico), ID_Bcn_2016 (vinculo con censos anteriores, falta en 685)
# QUE ES         Nom_Local, Nom_Activitat -> tipo_local
# DONDE          Nom_Barri y Nom_Districte (oficiales, cero nulos), Latitud/Longitud
# DIRECCION      Nom_Via + Num_Policia_Inicial, o Direccio_Unica ya montada
# EDIFICIO       Referencia_Cadastral agrupa locales del mismo inmueble
# CONTEXTO       SN_Mercat, SN_CComercial, SN_Galeria, SN_Eix: donde esta el local
# ATRIBUTOS      SN_Oci_Nocturn, SN_Obert24h, SN_Servei_Degustacio, SN_Mixtura, SN_Coworking
# CUANDO         Data_Revisio

print("=== LOS INDICADORES SN_, cuantos dicen que si ===")
for c in [x for x in d.columns if x.startswith("SN_")]:
    si = int(d[c].eq("Si").sum())
    print(f"  {c:<24} {si:>5,}  ({si / len(d):>5.1%})")
print()
print("=== AGRUPACIONES QUE PERMITEN ===")
print(f"  edificios distintos (Referencia_Cadastral): {d['Referencia_Cadastral'].nunique():,}"
      f" para {len(d):,} locales")
print(f"  ejes comerciales                          : {d['Nom_Eix'].nunique()}"
      f", con {int(d['Nom_Eix'].notna().sum()):,} locales dentro")
print(f"  mercados municipales                      : {d['Nom_Mercat'].nunique()}"
      f", con {int(d['Nom_Mercat'].notna().sum()):,} locales dentro")
print(f"  secciones censales                        : {d['Seccio_Censal'].nunique()}")


=== LOS INDICADORES SN_, cuantos dicen que si ===
  SN_Oci_Nocturn              50  ( 0.5%)
  SN_Coworking                 3  ( 0.0%)
  SN_Servei_Degustacio        80  ( 0.8%)
  SN_Obert24h                  1  ( 0.0%)
  SN_Mixtura                   7  ( 0.1%)
  SN_Carrer                9,261  (97.7%)
  SN_Mercat                   11  ( 0.1%)
  SN_Galeria                  24  ( 0.3%)
  SN_CComercial              193  ( 2.0%)
  SN_Eix                   3,095  (32.6%)

=== AGRUPACIONES QUE PERMITEN ===
  edificios distintos (Referencia_Cadastral): 7,952 para 9,481 locales
  ejes comerciales                          : 24, con 3,095 locales dentro
  mercados municipales                      : 10, con 11 locales dentro
  secciones censales                        : 181


In [10]:
# 10. De cuando es cada local
#
# El censo no es una foto de un dia: `Data_Revisio` dice cuando se visito cada local. Es la
# diferencia con OSM --alli la antiguedad tambien es desigual, pero el fichero no la trae--.
revision = pd.to_datetime(d["Data_Revisio"], errors="coerce")
print(f"trabajo de campo: {revision.min().date()} -> {revision.max().date()}")
print()
for anio, n in revision.dt.year.value_counts().sort_index(ascending=False).items():
    print(f"  {int(anio)}  {n:>6,}  ({n / len(d):>5.1%})")
print()
print("Reparto por tipo de local:")
display(pd.crosstab(d["tipo_local"], revision.dt.year))


trabajo de campo: 2021-10-18 -> 2024-10-17



  2024   4,088  (43.1%)
  2023   5,392  (56.9%)
  2021       1  ( 0.0%)

Reparto por tipo de local:


Data_Revisio,2021,2023,2024
tipo_local,,,
bar,1,2060,2212
comida_rapida,0,512,266
restaurante,0,2820,1610


## 5. El conjunto final

Solo lo que se va a usar: **identidad, nombre, donde esta y direccion**. Todo lo demas se queda en
el censo, que no se toca y sigue disponible si algun dia hace falta.

Dos columnas que no estaban en esa lista y aun asi entran, cada una por un motivo concreto:

- **`tipo_local`**, porque distinguir bar de restaurante de comida rapida es la razon por la que se
  eligieron esas tres categorias y no otras. Sin ella el filtro no se puede deshacer.
- **`fecha_revision`**, porque es la fecha del dato. Sin ella, el conjunto parece de 2024 entero
  cuando el 57% se visito en 2023.

In [11]:
# 11. Construccion del conjunto final
COLUMNAS = {
    "ID_Global": "local_id",
    "Nom_Local": "nombre",
    "tipo_local": "tipo_local",
    "Nom_Districte": "distrito",
    "Nom_Barri": "barrio",
    "Codi_Barri": "codigo_barrio",
    "Nom_Via": "calle",
    "Num_Policia_Inicial": "numero",
    "Direccio_Unica": "direccion",
    "Latitud": "latitud",
    "Longitud": "longitud",
    "Data_Revisio": "fecha_revision",
}

restauracion = d[list(COLUMNAS)].rename(columns=COLUMNAS)

# `SN` es como el censo escribe 'sin nombre'. Se pasa a nulo de verdad: dejarlo como texto haria
# que 55 locales se llamaran todos igual, y cualquier recuento por nombre los agruparia.
sin_nombre = restauracion["nombre"].astype(str).str.strip().str.upper().eq("SN")
restauracion.loc[sin_nombre, "nombre"] = pd.NA
restauracion["fecha_revision"] = pd.to_datetime(restauracion["fecha_revision"], errors="coerce")

print(f"{len(restauracion):,} locales x {restauracion.shape[1]} columnas")
print()
display(restauracion.head(8))


9,481 locales x 12 columnas



,local_id,nombre,tipo_local,distrito,barrio,codigo_barrio,calle,numero,direccion,latitud,longitud,fecha_revision
0,3e40cf2c-3b94-4977-af13-dae686ba63be2,THREE MARKS COFFEE,bar,Eixample,el Fort Pienc,5,AUSIÀS MARC,151,"028305, 151-151, LOC 10",41.397207,2.183138,2021-10-18
1,6a8c18de-61f6-44ec-9cc3-1b813a78d03b,NOT THE NAME JUST A COOL SIGN,restaurante,Eixample,la Dreta de l'Eixample,7,MÉNDEZ NÚÑEZ,17,"207803, 17-17, LOC 20",41.390123,2.177136,2023-04-04
2,c2ca40e1-5526-45c9-adf6-4ab9e701cf5f,BAR LES MATINADES,bar,Sants-Montjuïc,la Bordeta,16,PTGE ANDALUSIA,6,"700032, 6-8, LOC 10",41.370389,2.133569,2023-05-15
4,d1aaae88-f8c7-4576-b6b3-e02f6bffe147,ROLLING PIZZA,comida_rapida,Sants-Montjuïc,Sants - Badal,17,JUAN DE SADA,53,"173202, 53-53, LOC 20",41.378064,2.127568,2023-05-09
5,8395faa6-c4e2-465b-9752-2d56b4b9757f,TEMPURA - YA,restaurante,Eixample,l'Antiga Esquerra de l'Eixample,8,MUNTANER,153,"223606, 153-153, LOC 10",41.390957,2.153481,2023-04-12
6,ed4ca38d-09ba-437c-9205-b63c195ac566,CAL NINOT,bar,Eixample,l'Antiga Esquerra de l'Eixample,8,CASANOVA,136,"070607, 136-138, LOC 10",41.388684,2.154621,2023-04-11
7,bf8b59ac-60b3-480e-9096-969468972066,MAUR MUNTANER,restaurante,Eixample,l'Antiga Esquerra de l'Eixample,8,PROVENÇA,202,"268003, 202-202, LOC 20",41.389659,2.154958,2023-04-11
8,5f16c65a-22fa-4eca-aab4-350fa3a2d67b,GALICIA,restaurante,Eixample,la Nova Esquerra de l'Eixample,9,LLANÇA,38,"184402, 38-40, LOC 20",41.379205,2.146843,2023-04-21


In [12]:
# 12. Comprobaciones antes de dar por bueno el esquema
print("=== NULOS ===")
nulos = restauracion.isna().sum()
for c, n in nulos.items():
    marca = "" if n == 0 else f"   <- {n / len(restauracion):.1%}"
    print(f"  {c:<16} {n:>5}{marca}")
print()
print("=== IDENTIDAD ===")
print(f"  local_id unicos      : {restauracion['local_id'].nunique():,} de {len(restauracion):,}")
print(f"  nombres distintos    : {restauracion['nombre'].nunique():,}")
repetidos = restauracion["nombre"].value_counts()
print(f"  nombre mas repetido  : '{repetidos.index[0]}' en {repetidos.iloc[0]} locales"
      "   <- cadenas, no duplicados")
print()
print("=== TERRITORIO ===")
print(f"  barrios   : {restauracion['barrio'].nunique()} de 73")
print(f"  distritos : {restauracion['distrito'].nunique()} de 10")
mal_situados = ((restauracion.latitud < 41.30) | (restauracion.latitud > 41.48)
                | (restauracion.longitud < 2.05) | (restauracion.longitud > 2.24)
                | restauracion.latitud.isna())
print(f"  coordenadas nulas o fuera de Barcelona: {int(mal_situados.sum())}")
print()
print("=== REPARTO ===")
display(pd.crosstab(restauracion["distrito"], restauracion["tipo_local"], margins=True,
                    margins_name="TOTAL"))


=== NULOS ===
  local_id             0
  nombre              56   <- 0.6%
  tipo_local           0
  distrito             0
  barrio               0
  codigo_barrio        0
  calle                0
  numero               0
  direccion            0
  latitud              0
  longitud             0
  fecha_revision       0

=== IDENTIDAD ===
  local_id unicos      : 9,481 de 9,481
  nombres distintos    : 8,592
  nombre mas repetido  : 'SANDWICHEZ' en 24 locales   <- cadenas, no duplicados

=== TERRITORIO ===
  barrios   : 73 de 73
  distritos : 10 de 10
  coordenadas nulas o fuera de Barcelona: 0

=== REPARTO ===


tipo_local,bar,comida_rapida,restaurante,TOTAL
distrito,,,,
Ciutat Vella,402,167,660,1229
Eixample,990,172,1432,2594
Gràcia,276,69,289,634
Horta-Guinardó,304,21,136,461
Les Corts,185,24,241,450
Nou Barris,393,33,173,599
Sant Andreu,354,40,222,616
Sant Martí,690,96,498,1284
Sants-Montjuïc,421,69,414,904


## 6. Que queda pendiente

- **Pasar esto a un script** en `pipeline/gold/` y generar el `gold` desde el. El cuaderno queda
  como la justificacion de cada decision; el script, como lo que se ejecuta.
- **Que hacer fuera de Barcelona ciudad.** El censo solo cubre la ciudad. Para el resto de la
  provincia la unica fuente es OSM, con los problemas que se han visto, asi que la capa sera
  hibrida y hay que explicarlo en la web.

Cuando eso este cerrado: script en `pipeline/gold/` y `gold` generado desde el.
